Explore protocol paths and batch documents shared by Left and Right.


In [ ]:
from setup.nbs.inspect_tools import show, show_methods
from pair_recon_left.common.model.protocol import plan_path, mapping_path, unit_batch_path, result_batch_path
from pair_recon_left.common.model.protocol import unit_batch_document, result_batch_document
from pair_recon_left.common.model.protocol import list_plans, list_units, list_results, list_failures
from pair_recon_left.common.model.protocol import write_run_status, write_left_state, write_right_state
from pair_recon_left.common.model.protocol import refresh_units_status, col_check_close_document

show(plan_path, mapping_path, unit_batch_path, result_batch_path)
show(unit_batch_document, result_batch_document, list_plans, list_units, list_results, list_failures)
show(write_run_status, write_left_state, write_right_state, refresh_units_status, col_check_close_document)


In [ ]:
from pair_recon_left.common.model.protocol import plan_path, unit_batch_path, result_batch_path

# 路径函数是纯函数，可安全观察命名规则
print(plan_path('ORDERS'))
print(unit_batch_path(1, 64))
print(result_batch_path([{'unit_id': 'ORDERS:2026-01:AMOUNT'}]))


## Toy example

Run this cell locally to inspect the concrete inputs and outputs.


In [ ]:
from pprint import pprint
units = [{"unit_id": "ORDERS:2026-01:AMOUNT", "pair": "ORDERS", "window": "2026-01"}]
print("input units:", units)
pprint(unit_batch_document(1, units, left_ready_at="2026-01-01T00:00:00Z"))
pprint(result_batch_document([{"unit_id": units[0]["unit_id"], "verdict": "match"}], committed_at="2026-01-01T00:01:00Z"))


## Protocol reads and writes with an in-memory store

The fake store prints the keys and documents that would be persisted. It supports the same methods used by these helpers and performs no network calls.


In [ ]:
from pprint import pprint

class ToyStore:
    def __init__(self):
        self.documents = {}
        self.files = {}
    def exists(self, key):
        return key in self.documents
    def get_json(self, key):
        return self.documents[key]
    def put_json(self, key, value):
        self.documents[key] = value
        return key
    def list_keys(self, prefix):
        return sorted(key for key in self.documents if key.startswith(prefix))
    def put_bytes(self, key, body, *, content_type=None):
        self.files[key] = body
        return key

store = ToyStore()
print("plan path:", plan_path("ORDERS"))
print("mapping path:", mapping_path("ORDERS"))
print("initial plans:", list_plans(store))
print("initial units:", list_units(store))
print("initial results:", list_results(store))
print("initial failures:", list_failures(store))
print("run status key:", write_run_status(store, {"state": "running"}, timezone_name="UTC"))
print("left state key:", write_left_state(store, {"col_check_closed_at": "2026-01-01T00:00:00Z", "pairs": ["ORDERS"], "expected_units": 0}, timezone_name="UTC"))
print("right state key:", write_right_state(store, {"phase": "waiting"}, timezone_name="UTC"))
print("close document:"); pprint(col_check_close_document(store))
status_key = refresh_units_status(store)
print("units status key:", status_key)
print("units status CSV:", store.files[status_key].decode().strip())
print("stored JSON documents:"); pprint(store.documents)
